---
title: "Lecture 1: Introduction to Applied Statistics"
execute:
  enabled: true
jupyter: python3
---

## Three decisions, three datasets, one question

A hospital gets fined \$500K for "too many" readmissions. An Airbnb host wonders if they're underpricing by \$50/night. A pharmaceutical company must decide whether a \$2 billion drug actually works.

**How do you make these decisions with data?**

That's what this course is about. Not formulas — *decisions under uncertainty*.

> *"In God we trust; all others must bring data."* — W. Edwards Deming

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'https://raw.githubusercontent.com/stanford-mse-125/book/main/data'

## What is applied statistics?

Here's what this course is *not*: a parade of formulas, proofs, and toy examples with perfectly clean data.

Here's what it *is*: learning to make good decisions when the data is messy, the sample is limited, and the stakes are real.

**Applied statistics** is the science of making decisions under uncertainty using data. It sits at the intersection of:

- **Probability** (which you learned in MS&E 120) — the math of uncertainty
- **Computing** — because real datasets don't fit on a whiteboard
- **Domain knowledge** — because numbers without context are just numbers

As John Tukey put it: *"The best thing about being a statistician is that you get to play in everyone's backyard."* The same tools you'll learn in this course apply to healthcare, housing, sports, and drug development.

## The three acts of this course

This course follows a three-act structure, and the three datasets we just met map onto them:

**Act 1: Build Models** (Lectures 1–7) — Explore data, clean it, and build predictive models. We'll use regression, feature engineering, and decision trees on the Airbnb and hospital datasets.

**Act 2: Trust Models** (Lectures 8–13) — Bootstrap, hypothesis testing, regression inference, classification. We'll ask: how precise are our estimates? Is the drug effect real? Which coefficients are "real"?

**Act 3: See Further** (Lectures 14–19) — PCA, clustering, time series, tree-based methods in depth, causal inference. We'll move from "what happened" to "why."

Every week, we'll work with real data — with all the mess that entails.

## A first look: hospital readmissions

Let's start with a real dataset. The Centers for Medicare & Medicaid Services (CMS) tracks how often patients are readmitted to hospitals within 30 days of discharge. Hospitals with "too many" readmissions get fined — up to 3% of their Medicare payments.

In [ ]:
# Load hospital readmissions data
readmissions = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')
print(f"Shape: {readmissions.shape[0]:,} rows x {readmissions.shape[1]} columns")
readmissions.head(10)

Each row is one hospital-condition pair: a hospital's readmission performance for a specific condition (heart attack, pneumonia, heart failure, etc.). Let's see what conditions are tracked.

In [ ]:
readmissions['Measure Name'].value_counts()

The key column is the **Excess Readmission Ratio (ERR)**. CMS uses a statistical model that accounts for how sick each hospital's patients are. The ERR is the ratio of a hospital's predicted readmissions to its expected number, after adjusting for patient risk. A value above 1.0 means more readmissions than expected given the hospital's patient mix.

But here's the first lesson of applied statistics:

> **More data doesn't automatically mean better answers.** It depends on what's *in* the data — and what's missing.

In [ ]:
# Distribution of Excess Readmission Ratios
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(readmissions['Excess Readmission Ratio'].dropna(), bins=50, ax=ax,
             edgecolor='white')
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Expected = 1.0')
ax.axvline(x=1.05, color='orange', linestyle=':', linewidth=2, label='Your hospital = 1.05')
ax.set_xlabel('Excess Readmission Ratio')
ax.set_ylabel('Count')
ax.set_title('Hospital Readmission Performance Across the U.S.')
ax.legend()
plt.tight_layout()
plt.show()

Notice the distribution is centered near 1.0 — hospitals with ratios above 1.0 have more readmissions than expected, and those below have fewer. But there's real spread.

**Think about it:** If you ran a hospital and saw your ratio was 1.05 (the orange line), would you panic? How do you know if that's bad luck or a real problem? That's a statistics question.

## Another dataset: Airbnb pricing

Now a completely different question. You're an Airbnb host in New York City. You want to set your price. Too high and nobody books; too low and you leave money on the table. What's the right price?

In [ ]:
# Load Airbnb data (just a few key columns for now)
airbnb = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False,
                     usecols=['name', 'neighbourhood_group_cleansed', 'room_type',
                              'price', 'bedrooms', 'number_of_reviews'])

# Clean price column (may contain $ and commas in raw Airbnb data)
airbnb['price'] = airbnb['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)
print(f"{airbnb.shape[0]:,} listings in NYC")
airbnb.head()

In [ ]:
airbnb['price'].describe()

Look at the output above: the mean and the max. And there are listings near \$0. Already the data is telling us something: the "average" might not be very meaningful here. Extreme values — \$0 listings that aren't real prices, and sky-high outliers — distort the mean. *Outliers distort averages* — that's a theme we'll return to all quarter.

In [ ]:
# Price distribution — notice the long right tail
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(airbnb['price'].dropna(), bins=100, ax=ax, edgecolor='white')
ax.set_xlabel('Price per night ($)')
ax.set_ylabel('Count')
ax.set_title('NYC Airbnb Price Distribution')
ax.axvline(airbnb['price'].median(), color='orange', linestyle='--', lw=2,
           label=f'Median = ${airbnb["price"].median():.0f}')
ax.axvline(airbnb['price'].mean(), color='red', linestyle='--', lw=2,
           label=f'Mean = ${airbnb["price"].mean():.0f}')
ax.set_xlim(0, 1000)
ax.legend()
plt.tight_layout()
plt.show()

Notice how far the mean (red) is pulled to the right by expensive listings. The median is a better summary here. We'll dive deep into this data in Lecture 2.

**Think about it:** If you had a 1-bedroom apartment in Manhattan, would you price at the mean? Why or why not? What other information would you want?

## What happens when you ask an AI to analyze data?

So you have these datasets. You could spend hours exploring them — or you could hand them to an AI assistant and get an analysis in seconds. Let's see what happens.

Imagine you give ChatGPT (or Claude, or Copilot) the hospital dataset and say: *"Analyze this data and tell me which hospitals have the worst readmission rates."*

### What the AI gets right

- **Fast EDA**: it would compute summary statistics, make histograms, identify the shape of the data — all in seconds
- **Decent plots**: bar charts of top/bottom hospitals, distributions by condition
- **Code that runs**: the pandas/matplotlib code would be syntactically correct

### What the AI gets wrong

- **No skepticism about the data**: it wouldn't ask "why are 15% of the values missing?" It would silently drop those rows.
- **No domain context**: it wouldn't know that "Too Few to Report" means small/rural hospitals — and that dropping them biases the analysis toward large urban hospitals.
- **Plausible nonsense**: it could rank hospitals by raw readmission count instead of the risk-adjusted ERR, penalizing hospitals that treat sicker patients. The ranking would *look* reasonable but be *wrong*.

In [ ]:
# Here's what "Too Few to Report" looks like in the data
non_numeric = readmissions[pd.to_numeric(readmissions['Number of Readmissions'],
                                         errors='coerce').isna()]
print(f"Rows with non-numeric readmission counts: {len(non_numeric):,}")
print(f"That's {len(non_numeric)/len(readmissions)*100:.1f}% of the data")
print()
print("What values do they have?")
print(non_numeric['Number of Readmissions'].value_counts())

This is your first example of a **missing data mechanism** — the data isn't missing randomly, it's missing because of a systematic pattern (small hospitals don't have enough cases to report). An AI would drop these rows without mentioning it. But *dropping them changes the answer* — it biases your analysis toward large urban hospitals.

In [ ]:
# Which hospitals have "Too Few to Report"? Let's look at a sample.
cols_to_show = ['Facility Name', 'State', 'Measure Name', 'Number of Readmissions']
available_cols = [c for c in cols_to_show if c in non_numeric.columns]
non_numeric[available_cols].head(8)

In [ ]:
# How many hospitals are affected, by condition?
fig, ax = plt.subplots(figsize=(8, 5))
(non_numeric['Measure Name']
 .value_counts()
 .plot.barh(ax=ax, color='C3', edgecolor='white'))
ax.set_xlabel('Number of hospitals with "Too Few to Report"')
ax.set_title('Missing data is NOT random — some conditions are harder to track')
plt.tight_layout()
plt.show()

**That's the kind of thing you'll learn to catch in this course.** AI tools can write code fast, but they lack skepticism and domain knowledge. Your job is to provide both.

## The \$2 billion question

The third scenario we opened with — does this drug work? — is perhaps the highest-stakes application of statistics.

In a clinical trial, you randomly assign patients to get the drug or a placebo. You measure outcomes. Then you ask: is the difference between the groups real, or could it be due to chance?

This is **hypothesis testing**, and we'll spend several weeks on it in Act 2. The logic in brief:

1. Assume the drug does nothing (the "null hypothesis")
2. Ask: if the drug does nothing, how likely is it that we'd see a difference this large?
3. If the answer is "very unlikely" — below a threshold we set in advance, called the **significance level** — we reject the null hypothesis

We won't analyze clinical trial data today. But notice: it's the same logic whether you're testing a drug, comparing hospitals, or checking if an Airbnb price is unusual. The framework is universal — the context changes everything.

> *"Far better an approximate answer to the right question, which is often vague, than an exact answer to the wrong question, which can always be made precise."* — John Tukey

**Think about it:** In the hospital example, what would the null hypothesis be? (Hint: "This hospital's readmission rate is no different from expected.")

## What you'll be able to do by the end

By the end of MSE 125, you'll be able to:

1. **Explore** a dataset and identify problems before they ruin your analysis
2. **Model** relationships in data using regression and classification
3. **Quantify uncertainty** — not just give a number, but say how confident you are
4. **Reason about causation** — not just correlation
5. **Critically evaluate** statistical claims — including those made by AI tools

That last point is important. AI assistants are getting very good at writing code and producing analyses. But they have no judgment. They can't tell you when an analysis is misleading, when the data is biased, or when the conclusions don't follow from the evidence.

**By the end of this course, you'll be able to tell when the AI is wrong — and why.**

## Coming up next

In **Lecture 2**, we'll roll up our sleeves and do exploratory data analysis (EDA) on the hospital and Airbnb datasets. We'll look at distributions, missing data, and outliers — and find a surprise hiding in the hospital data.

In **Lecture 3**, we'll tackle data munging and put an AI assistant to the test on real data.

And in the final weeks of the course, we'll return to this hospital data and ask a harder question: does a hospital's readmission rate *cause* its penalty, or is something else going on?

## Key Takeaways

- **Applied statistics is about decisions under uncertainty**, not formulas in a vacuum.
- Real data is messy: missing values, outliers, confounding variables. That mess is the point.
- AI tools can write code fast, but they lack skepticism and domain knowledge. Your job is to provide both.
- This course has three acts: Explore, Model, Infer. Each builds on the last.
- Every dataset has a story. Learning to read that story — and question it — is the core skill of a statistician.

## Study guide

### Key definitions

- **Applied statistics**: the science of making decisions under uncertainty using data.
- **Excess Readmission Ratio (ERR)**: CMS measure comparing a hospital's readmissions to what's expected given its patient mix. Above 1.0 = more readmissions than expected.
- **"Too Few to Report"**: a missing data mechanism in the hospital dataset — small hospitals don't have enough cases, so their data is suppressed. Not missing at random.
- **Null hypothesis** (preview): the default assumption that nothing interesting is happening (e.g., "this hospital is no different from average"). Formalized in Lectures 9–10.

### Key ideas (one sentence each)

1. More data doesn't automatically mean better answers — it depends on what's in the data and what's missing.
2. Missing data is often a signal, not just a nuisance — *why* data is missing matters as much as *that* it's missing.
3. AI tools produce plausible-looking analyses fast, but lack the skepticism and domain knowledge to catch misleading results.
4. Outliers distort averages — always look at the distribution, not just summary statistics.

### Computational tools

- `pd.read_csv()` — load a CSV file into a DataFrame
- `.head()` — peek at the first few rows
- `.describe()` — summary statistics (mean, std, min, max, quartiles)
- `.value_counts()` — count unique values in a column
- `sns.histplot()` — plot a histogram
- `pd.to_numeric(errors='coerce')` — convert to numeric, turning non-numbers into NaN

### For the quiz

- Know what the Excess Readmission Ratio measures and what a value above 1.0 means.
- Be able to explain why silently dropping "Too Few to Report" rows biases an analysis.
- Understand the three-act structure of the course (Explore, Model, Infer).